# 🇪🇬 Fine-Tuning Qwen2.5-3B for Egyptian Document OCR Extraction
### Trained on authentic Egyptian Birth Certificates, National IDs, and Utility Bills

This notebook fine-tunes **Qwen/Qwen2.5-3B-Instruct** using 4-bit QLoRA on a free Google Colab T4 GPU (takes ~8 minutes).

---
### How to run:
1. Switch to GPU: **Runtime** -> **Change runtime type** -> select **T4 GPU** -> **Save**.
2. On the left sidebar, click the **📁 Files** icon, click the **Upload** button, and upload:
   - `egypt_docs_train.jsonl`
   - `egypt_docs_val.jsonl`
   *(Both files are already generated in your local `sahelha-backend/backend/data/` folder)*
3. Click **Runtime** -> **Run all**.
4. When finished, the last cell will automatically download `qwen2.5-3b-egypt-adapter.zip` to your computer!

In [ ]:
# Step 1: Install fine-tuning dependencies
!pip install -q -U torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q -U transformers datasets peft bitsandbytes accelerate trl

In [ ]:
# Step 2: Verify GPU
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    raise SystemError("Please enable GPU: Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# Step 3: Check uploaded dataset files (or auto-create data folder)
import os, glob
train_candidates = glob.glob("**/*train.jsonl", recursive=True)
val_candidates = glob.glob("**/*val.jsonl", recursive=True)

if not train_candidates or not val_candidates:
    print("\n[!] Dataset files not found in root. Please drag and drop 'egypt_docs_train.jsonl' and 'egypt_docs_val.jsonl' into the Colab Files panel on the left!")
    from google.colab import files
    print("Or upload them now using the button below:")
    uploaded = files.upload()
    train_path = "egypt_docs_train.jsonl"
    val_path = "egypt_docs_val.jsonl"
else:
    train_path = train_candidates[0]
    val_path = val_candidates[0]
    print(f"Found Train Dataset: {train_path}")
    print(f"Found Val Dataset:   {val_path}")

In [ ]:
# Step 4: 4-bit QLoRA Fine-Tuning Qwen2.5-3B
import json, gc, os, torch
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    Trainer, TrainingArguments, DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

model_id = "Qwen/Qwen2.5-3B-Instruct"
max_seq_length = 384

# 1. 4-bit NormalFloat Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print(f"Loading tokenizer for {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Loading 4-bit base model {model_id}...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

# 2. LoRA Config targeting attention projections
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 3. Dataset Preprocessing
def load_chatml_data(file_path):
    records = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append({"messages": json.loads(line)["messages"]})
    raw = Dataset.from_list(records)
    def tokenize_fn(ex):
        texts = [tokenizer.apply_chat_template(m, tokenize=False) for m in ex["messages"]]
        toks = tokenizer(texts, truncation=True, max_length=max_seq_length, padding="max_length")
        toks["labels"] = toks["input_ids"].copy()
        return toks
    return raw.map(tokenize_fn, batched=True, remove_columns=["messages"])

print("Preprocessing datasets...")
train_ds = load_chatml_data(train_path)
val_ds = load_chatml_data(val_path)

# 4. Training Arguments
training_args = TrainingArguments(
    output_dir="./qwen2.5-3b-egypt-adapter",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt"),
)

print("Starting training on T4 GPU (approx. 8 minutes)...")
trainer.train()

trainer.model.save_pretrained("./qwen2.5-3b-egypt-adapter")
tokenizer.save_pretrained("./qwen2.5-3b-egypt-adapter")
print("Training complete! Adapter saved to ./qwen2.5-3b-egypt-adapter")

In [ ]:
# Step 5: Test the Fine-Tuned Model on the Egyptian Birth Certificate
SYSTEM_PROMPT = "You are an expert Arabic document analyst specialized in Egyptian government documents and administrative paperwork. Return ONLY valid JSON."

test_ocr = """جمهورية مصر العربية
وزارة الداخلية
قطاع مصلحة الأحوال المدنية
www.cso.gov.eg
صورة قيد الميلاد
الرقم القومي : ۳۱۵۰۳۰۷۰۱۰۷۷۰۸
بيانات المولود
خديجة
الجنسية : مصر    الديانة : مسلمة
النوع : انثى
محل الميلاد : القاهرة / السيدة زينب
تاريخ الميلاد : سبعه من مارس عام الفان و خمسه عشر
بيانات الأب
سامح سمير فضل اللبودى
الرقم القومي : ٢٨٦٠١١٥١١٠٠٢٧٨    الديانة : مسلم
الجنسية : مصر
بيانات الأم
مى انور هارون بخات
الرقم القومي : ٢٨٨٠٢١٨٨٨٠٠٥٤٢    الديانة : مسلمة
الجنسية : مصر
م . صحة : السيدة ثان    رقم القيد : ١٢٥٠
س . مدني : السيدة زينب    ت . القيد : ٢٠١٥/٠٣/١٢
س . اصدار : سجل الخليفة    ت . اصدار : ٢٠١٥/١١/٢٥
رقم مسلسل : ١٨٤٩٢١٦١٤٣"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": test_ocr},
]
inputs = tokenizer(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True), return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.1)
response = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
print("\n=== FINE-TUNED MODEL PREDICTION ===\n")
print(response)

In [ ]:
# Step 6: Zip and Download LoRA Adapter to your local computer
!zip -r qwen2.5-3b-egypt-adapter.zip ./qwen2.5-3b-egypt-adapter
from google.colab import files
print("Triggering browser download...")
files.download("qwen2.5-3b-egypt-adapter.zip")